# Comparison of models to test explanation of CE signatures

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [ ]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join('data', 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join('data', 'final-document.tsv')

In [ ]:
df = pd.read_table(FINAL_DOC, sep='\t')
print(df.shape)
df.head()

In [ ]:
df['resid2'] = df['resid']**2

In [ ]:
df['conversation_id'].nunique()

## LME Regression: Predicting CE

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
test_var = [
    # 'x_conv_leader',
    'x_in_common',
    'x_responsive_1',
    'x_responsive_2',
    'x_responsive_3',
    'x_our_thoughts_synced_up_sr1',
    'x_developed_joint_perspective_sr2',
    'x_thoughts_became_more_alike_sr5',
    'x_saw_world_in_same_way_sr8',
    'x_you_are_good_listener'
]

In [ ]:
data = {'params': [], 'aic': []}

In [ ]:
for var in tqdm(test_var):
    ######## linear model
    model = "{} ~ resid".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    # aic table
    aic_result = {
        'var': var, 
        'linear': (2*len(mdf.params)) - (2 * mdf.llf)
    }
    
    ######## square model
    model = "{} ~ resid2".format(var)
    md = smf.glm(model,data=df)
    # md = smf.ols(model,data=df)
    mdf = md.fit()
    
    # model parameters table
    result = {'var': var}
    for p in mdf.params.index:
        result[p] = '{} ({})'.format(
            np.format_float_scientific(mdf.params[p], precision=2), 
            np.format_float_scientific(mdf.bse[p], precision=2)
        ) + ''.join(['*'] * (int(mdf.pvalues[p] < .05) + int(mdf.pvalues[p] < .01) + int(mdf.pvalues[p] < .00001) ))
    # result['r2'] = mdf.rsquared
    data['params'] += [result]
    
    aic_result['squared'] = (2*len(mdf.params)) - (2 * mdf.llf)
    
    data['aic'] += [aic_result]
    
data['params'] = pd.DataFrame(data['params'])
data['aic'] = pd.DataFrame(data['aic'])

### Looking at models and parameters

In [ ]:
data['params'] = data['params'].transpose()
data['params']['var'] = data['params'].index
data['params'] = data['params'][['var'] + [col for col in data['params'].columns if str(col) != 'var']]

In [ ]:
data['params'].head()

In [ ]:
search_cols = np.array([col for col in np.unique(data['params'].values[0,:]) if col != 'var'])
search_cols

In [ ]:
d = []

In [ ]:
for col in search_cols:
    sel = (data['params'].values[0,:] == col).nonzero()[0]-1
    vs = data['params'][['var']+sel.tolist()].values
    d += [vs]
d = np.concatenate(d,axis=0)
d = pd.DataFrame(d, columns=['var', '(1)', '(2)'])

In [ ]:
sel = d['var'].isin(['var'])
d['var'].loc[sel] = d['(1)'].loc[sel]

In [ ]:
data['params']=d

In [ ]:
data['params'].head(n=100)

In [ ]:
fin_data_path = os.path.join('data', 'parameter-estimates.csv')

In [ ]:
data['params'].to_csv(fin_data_path,  header=False, encoding='utf-8')

In [ ]:
with open(fin_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['params'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()

### Comparing AIC scores

In [ ]:
aic_data_path = os.path.join('data', 'aic.csv')

In [ ]:
data['aic']['model_sel'] = data['aic']['linear'] - data['aic']['squared']
data['aic'].head(n=100)

In [ ]:
data['aic'].to_csv(aic_data_path, index=False, encoding='utf-8')

In [ ]:
with open(aic_data_path.replace('.csv', '.txt'), 'w') as f:
    txt =  data['aic'].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()